# Draft search walkthrough

---

This notebook follows the same steps as `notebooks/semantic_search_starter_notebook.ipynb`. The starter uses three toy sentences and an in-memory database. These cells use the 2025 NFL draft and save Chroma to disk.

1. Pick a dataset
2. Chunk + embed your data
3. Store your data in a Chroma vector db
4. Build a search app using Gradio
5. The saved files that get published

The source file is the full nflverse draft table:

https://github.com/nflverse/nflverse-data/releases/download/draft_picks/draft_picks.parquet

That file starts in 1980. `load_draft_picks(2025)` keeps one season. The saved copy of those 257 rows is `data/raw/draft-picks-2025.csv`.

Install once, from the project folder:

`pip install -r requirements.txt`

## 1. Load the embedding model

The embedding model. The first run downloads it. Later runs reuse the local copy. The build script and `app.py` must keep this exact name.

In [1]:
# Same MiniLM name the app uses.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


## 2. Pick a dataset

Read the raw table. Each row is one drafted player. Position is still a short code such as `QB`.

In [2]:
import polars as pl
from pathlib import Path

# Data and Chroma live in the project root, not in notebooks/.
root = Path.cwd()
if not (root / "data" / "raw" / "draft-picks-2025.csv").exists():
    root = root.parent

raw_path = root / "data" / "raw" / "draft-picks-2025.csv"
draft = pl.read_csv(raw_path)

print(draft.height)
print(
    draft.select(
        ["season", "round", "pick", "team", "pfr_player_name", "position", "college"]
    ).head(3)
)


257
shape: (3, 7)
┌────────┬───────┬──────┬──────┬─────────────────┬──────────┬────────────┐
│ season ┆ round ┆ pick ┆ team ┆ pfr_player_name ┆ position ┆ college    │
│ ---    ┆ ---   ┆ ---  ┆ ---  ┆ ---             ┆ ---      ┆ ---        │
│ i64    ┆ i64   ┆ i64  ┆ str  ┆ str             ┆ str      ┆ str        │
╞════════╪═══════╪══════╪══════╪═════════════════╪══════════╪════════════╡
│ 2025   ┆ 1     ┆ 1    ┆ TEN  ┆ Cam Ward        ┆ QB       ┆ Miami (FL) │
│ 2025   ┆ 1     ┆ 2    ┆ JAX  ┆ Travis Hunter   ┆ WR       ┆ Colorado   │
│ 2025   ┆ 1     ┆ 3    ┆ NYG  ┆ Abdul Carter    ┆ DE       ┆ Penn St.   │
└────────┴───────┴──────┴──────┴─────────────────┴──────────┴────────────┘


## 3. Chunk

One pick is already one fact, so we do not split the text further. The chunk is one sentence.

The model does not treat `QB` as the word quarterback, so the sentence uses the full word. The position word goes first because the rest of the sentence looks the same for every player.

In [3]:
# Full words. MiniLM does not treat QB as quarterback.
POSITION_WORDS = {
    "QB": "quarterback",
    "RB": "running back",
    "WR": "wide receiver",
    "TE": "tight end",
    "OL": "offensive lineman",
    "OT": "offensive tackle",
    "OG": "offensive guard",
    "G": "offensive guard",
    "C": "center",
    "DL": "defensive lineman",
    "DE": "defensive end",
    "DT": "defensive tackle",
    "LB": "linebacker",
    "OLB": "outside linebacker",
    "DB": "defensive back",
    "CB": "cornerback",
    "S": "safety",
    "SAF": "safety",
    "FS": "free safety",
    "K": "kicker",
    "P": "punter",
    "LS": "long snapper",
}

ROUND_WORDS = {
    1: "first",
    2: "second",
    3: "third",
    4: "fourth",
    5: "fifth",
    6: "sixth",
    7: "seventh",
}

texts = []
ids = []
for row in draft.iter_rows(named=True):
    name = row["pfr_player_name"] or "Unknown player"
    code = row["position"] or ""
    college = row["college"] or "unknown college"
    team = row["team"] or "an unknown team"
    position = POSITION_WORDS.get(code, code or "unknown position")
    round_word = ROUND_WORDS.get(row["round"], "unknown")

    # Position first, so it is not lost in the shared sentence.
    sentence = (
        f"{position}. {name} is a {position} from {college}. "
        f"{team} drafted him in the {round_word} round of the "
        f"{row['season']} NFL draft, pick {row['pick']}."
    )
    texts.append(sentence)
    ids.append(f"{row['season']}-{row['pick']}")

processed_path = root / "data" / "processed" / "indexed-draft-sentences-2025.csv"
pl.DataFrame({"id": ids, "document": texts}).write_csv(processed_path)

print(len(texts))
print(texts[0])


257
quarterback. Cam Ward is a quarterback from Miami (FL). TEN drafted him in the first round of the 2025 NFL draft, pick 1.


## 4. Embed

`encode` turns each sentence into a vector of 384 numbers. `normalize_embeddings=True` makes each vector length 1. Cosine distance needs that. A smaller distance means a closer match.

In [4]:
# Same normalize flag as search.
embeddings = model.encode(texts, normalize_embeddings=True)
print(embeddings.shape)


(257, 384)


## 5. Store in Chroma

An in-memory database disappears when the notebook stops. `PersistentClient` writes `chromadb/chroma.sqlite3` so `app.py` can open it later.

Running this cell again replaces the `draft_2025` collection. Same ids would be skipped, not updated.

In [5]:
import chromadb
from chromadb.config import Settings
from pathlib import Path

root = Path.cwd()
if not (root / "data" / "raw" / "draft-picks-2025.csv").exists():
    root = root.parent

client = chromadb.PersistentClient(
    path=str(root / "chromadb"),
    settings=Settings(anonymized_telemetry=False),
)

# Delete first. Re-adding the same ids would skip the new text.
try:
    client.delete_collection("draft_2025")
except Exception:
    pass

collection = client.create_collection(
    "draft_2025",
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=ids,
)

print(collection.count())


257


## 6. Search

Embed the question with the same model, then ask Chroma for the nearest sentences. The starter returns 1 hit. Five hits make the result easier to judge.

In [6]:
def semantic_search(query):
    query = (query or "").strip()
    if not query:
        return "Type a search, such as: quarterback."

    query_embedding = model.encode([query], normalize_embeddings=True)
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=5,
    )

    lines = []
    for document, distance in zip(results["documents"][0], results["distances"][0]):
        lines.append(f"{distance:.3f} — {document}")
    return "\n\n".join(lines)


print(semantic_search("quarterback"))


0.365 — quarterback. Cam Ward is a quarterback from Miami (FL). TEN drafted him in the first round of the 2025 NFL draft, pick 1.

0.388 — quarterback. Tyler Shough is a quarterback from Louisville. NOR drafted him in the second round of the 2025 NFL draft, pick 40.

0.415 — quarterback. Jalen Milroe is a quarterback from Alabama. SEA drafted him in the third round of the 2025 NFL draft, pick 92.

0.422 — quarterback. Graham Mertz is a quarterback from Florida. HOU drafted him in the sixth round of the 2025 NFL draft, pick 197.

0.434 — quarterback. Jaxson Dart is a quarterback from Mississippi. NYG drafted him in the first round of the 2025 NFL draft, pick 25.


## 7. Gradio

One text box in, one text box out. Same shape as the starter. This cell opens a local page. If port 7860 is already in use, stop the other `app.py` first.

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(label="Enter your search query"),
    outputs=gr.Textbox(label="Top Matches"),
    title="Semantic Search Engine",
    description=(
        "Search the 2025 NFL draft. "
        "Source: https://github.com/nflverse/nflverse-data/releases/download/draft_picks/draft_picks.parquet. "
        "A smaller distance is a closer match."
    ),
    examples=["quarterback", "wide receiver", "kicker", "defensive end"],
)

demo.launch()


## 8. Where those steps landed

The notebook cells above are the same work as `scripts/build-draft-index.py` and `app.py`. Settings shared by both files live in `draft_search_settings.py`.

- `data/raw/draft-picks-2025.csv` is the dataset cell
- `data/processed/indexed-draft-sentences-2025.csv` is the chunk cell
- `chromadb/chroma.sqlite3` is the store cell
- `app.py` is the search function plus the Gradio cell, loading the saved database instead of building it
- `requirements.txt` is the install line
- `README.md` is the Hugging Face Space config and the folder map

Run the app outside the notebook with `python app.py`.